# PHYS 405 Computerized Data Analysis · Week 4 · Moments and moment generating functions
Instructor: Prof. Dr. Bora Işıldak · Department of Physics, Yeditepe University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/Phys_405_2026_2027_Fall/blob/main/Labs/Labs4/Lab4.ipynb)

**File → Save a copy in Drive, then work in your copy.** Your work lives in your Drive copy, not in the link.

How the lab works (eight problems, about 14 minutes each):
1. Read the problem. Write your prediction in the **Predict** cell.
2. Run the code, then **Compare** with your prediction.
3. Run the two checks. Each check tests the answer a different way.
4. Do the **Your turn** task in the empty cell.

Today's physics: ion charge states, a 1 GHz oscillator, scintillator pulse heights, pion decay times, a spin-1 beam, hit times, ADC rounding errors and the π → μ → e decay chain. All simulated numbers use `np.random.default_rng(seed)` with the seed given in each problem, so your printed numbers match the slides.
The lab uses new data: the lecture's numbers do not carry over.

**Your Drive folder.** The next cell connects Google Drive (Colab asks for permission once) and makes the folder
`MyDrive/PHY405_Labs/Week04` if it does not exist. Every file this notebook writes goes there.

In [ ]:
import os
try:
    from google.colab import drive                      # running in Colab
    drive.mount("/content/drive")
    LAB_DIR = "/content/drive/MyDrive/PHY405_Labs/Week04"
except ImportError:                                     # running elsewhere: a local folder
    LAB_DIR = os.path.join(os.getcwd(), "PHY405_Labs", "Week04")
os.makedirs(LAB_DIR, exist_ok=True)
os.chdir(LAB_DIR)                                       # files written below land here
print("working folder:", os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'
print("numpy", np.__version__)

## Problem 1 · Moments of a discrete variable
An ion source gives charge states $q = 1, 2, 3, 4$ with fractions 0.4, 0.3, 0.2, 0.1 (illustrative). Find $E[q]$, $E[q^2]$ and $\operatorname{Var}(q)$.

*Givens:* $E[q^n] = \sum q^n P(q)$; $\operatorname{Var} = E[q^2] - \mu^2$.

**Predict.** Is $E[q^2]$ the same number as $(E[q])^2$? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
q, p = np.array([1, 2, 3, 4]), np.array([0.4, 0.3, 0.2, 0.1])
mu, m2 = (q * p).sum(), (q**2 * p).sum()
print(round(mu, 3), round(m2, 3), round(m2 - mu**2, 3))
rng = np.random.default_rng(21)
s = rng.choice(q, 100_000, p=p)
print(round(s.mean(), 3), round(s.var(), 3))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The trap $(E[q])^2 = 4.0$ would give a variance of 0.

In [ ]:
# check A (definition): the central moment sum((q - mu)^2 p) gives the same variance
assert abs(((q - mu)**2 * p).sum() - (m2 - mu**2)) < 1e-12
print("check A passed")

In [ ]:
# check B (simulation): 100 000 ions (seed 21) agree with the formulas to 0.02
assert abs(s.mean() - mu) < 0.02 and abs(s.var() - (m2 - mu**2)) < 0.02
print("check B passed")

**Your turn.** A particle hits a 2 m bar at a uniformly random position $x$. Find $E[x]$, $E[x^2]$, $\operatorname{Var}(x)$ and $\sigma$.

*Hint:* the density is $1/(2\ \text{m})$ on $[0, 2\ \text{m}]$; $E[x^2] = \int x^2 f(x)\,dx$.

In [ ]:
# TODO: write your answer here

## Problem 2 · A variance that cancels away
A counter reads a 1 GHz oscillator 1000 times; the readings scatter by $\sigma = 0.5$ Hz (simulated). Compute the variance with `mean(f**2) - mean(f)**2`, then with `np.var`. Do they agree?

*Givens:* $f \approx 10^9$ Hz; float64 keeps about 16 significant digits; seed 1.

**Predict.** Do both formulas give about 0.25 Hz²? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
rng = np.random.default_rng(1)
f = 1e9 + rng.normal(0, 0.5, 1000)          # Hz
print(np.mean(f**2) - np.mean(f)**2)        # one pass
print(round(np.var(f), 3))                  # mean first
g = f - 1e9                                 # shift the origin
print(round(np.mean(g**2) - np.mean(g)**2, 3))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. Can a variance be negative? Near $10^{18}$ Hz² the float64 numbers are `np.spacing(1e18)` apart: print it.

In [ ]:
# check A (sign): a variance is a mean of squares, so the one-pass number is wrong, not the physics
one_pass = np.mean(f**2) - np.mean(f)**2
assert one_pass < 0 <= np.var(f)
print("check A passed; spacing of doubles near 1e18:", np.spacing(1e18))

In [ ]:
# check B (independent method): shifting the origin (the parallel-axis theorem) gives np.var
assert abs((np.mean(g**2) - np.mean(g)**2) - np.var(f)) < 1e-6
print("check B passed")

**Your turn.** A DAQ stores temperatures near 300 K as float32; the true scatter is $\sigma = 1$ mK. Simulate 1000 readings with seed 9, convert with `.astype(np.float32)`, and compute the variance both ways. What does the one-pass formula give?

*Hint:* float32 keeps about 7 digits; subtract 300 K before squaring.

In [ ]:
# TODO: write your answer here

## Problem 3 · Skewness from data
5000 pulse heights from a thin scintillator (simulated Moyal shape, ADC counts). Compute the mean, the median and the sample skewness. Are most pulses above the mean?

*Givens:* $g_1 = m_3 / m_2^{3/2}$, with $m_k$ the mean of $(x - \bar x)^k$; seed 3.

**Predict.** The skewness is positive. Are most pulses above the mean? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
from scipy import stats
rng = np.random.default_rng(3)
x = stats.moyal.rvs(loc=40, scale=5, size=5000, random_state=rng)
d = x - x.mean()
g1 = np.mean(d**3) / np.mean(d**2)**1.5
print(round(x.mean(), 1), round(np.median(x), 1), round(g1, 2))
print(round(np.mean(x > x.mean()), 2), round(float(stats.moyal.stats(moments='s')), 3))

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.hist(x, bins=np.arange(20, 132.1, 4.0), color="tab:blue")
plt.axvline(x.mean(), color="tab:orange", label="mean")
plt.axvline(np.median(x), color="tab:green", ls="--", label="median")
plt.xlabel("pulse height (ADC)"); plt.ylabel("pulses per 4 ADC"); plt.legend(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. Which way does the tail point, and where is the mean relative to the median?

In [ ]:
# check A (order): a right tail puts the mean above the median
assert x.mean() > np.median(x)
print("check A passed")

In [ ]:
# check B (independent value): the sample skewness is within 0.1 of the exact Moyal skewness
assert abs(g1 - float(stats.moyal.stats(moments='s'))) < 0.1
print("check B passed")

**Your turn.** For the same 5000 pulses, compute the excess kurtosis $m_4/m_2^2 - 3$. Is it larger than a Gaussian's (0)?

*Hint:* `np.mean(d**4)` with the same `d`.

In [ ]:
# TODO: write your answer here

## Problem 4 · Moments of a density
A stopped π⁺ decays with mean life $\tau = 26.033$ ns. Compute $E[T]$, $E[T^2]$, $E[T^3]$ numerically from the density and compare with $n!\,\tau^n$. Then the skewness.

*Givens:* $f(t) = e^{-t/\tau}/\tau$; `np.trapezoid` on a fine grid; integrate far enough.

**Predict.** Is it enough to integrate over 0 to $2\tau$, where most decays are? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
tau = 26.033                                  # ns, charged pion
t = np.linspace(0, 20 * tau, 200_001)
f = np.exp(-t / tau) / tau
E = [np.trapezoid(t**n * f, t) for n in (1, 2, 3)]
print([float(round(e / tau**n, 3)) for e, n in zip(E, (1, 2, 3))])
s2 = E[1] - E[0]**2
print(round((E[2] - 3*E[0]*E[1] + 2*E[0]**3) / s2**1.5, 3))

In [ ]:
plt.figure(figsize=(7, 3.5))
u = t / tau
for n, c in ((1, "tab:blue"), (2, "tab:green"), (3, "tab:orange")):
    plt.plot(u, u**n * np.exp(-u), color=c, label=f"$t^{n} f(t)$ (units of $\\tau^{n-1}$)")
plt.axvline(2, color="tab:red", ls="--", label="2τ")
plt.xlim(0, 10); plt.xlabel("t/τ"); plt.legend(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The trap: stop at $2\tau$. Compute `np.trapezoid` on `t[t <= 2*tau]` and compare with $\tau$.

In [ ]:
# check A (exact values): E[T^n] = n! tau^n
assert np.allclose([E[0] / tau, E[1] / tau**2, E[2] / tau**3], [1, 2, 6], atol=1e-3)
print("check A passed")

In [ ]:
# check B (identity): the skewness of every exponential is 2
assert abs((E[2] - 3*E[0]*E[1] + 2*E[0]**3) / s2**1.5 - 2) < 1e-3
print("check B passed")

**Your turn.** Compute the root-mean-square decay time $\sqrt{E[T^2]}$ of the pion. Is it longer or shorter than $\tau$?

*Hint:* $E[T^2]$ is the second number your code computed; $E[T^2] = 2\tau^2$.

In [ ]:
# TODO: write your answer here

## Problem 5 · An MGF by hand
A spin-1 beam has $m = -1, 0, +1$ with probabilities 0.2, 0.3, 0.5 (illustrative). Write $M(t)$ and find $E[m]$ and $\operatorname{Var}(m)$ from its derivatives.

*Givens:* $M(t) = \sum_m P(m)\,e^{tm}$; $M'(0) = E[m]$, $M''(0) = E[m^2]$.

**Predict.** Is the variance $M''(0)$? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
m, p = np.array([-1, 0, 1]), np.array([0.2, 0.3, 0.5])
M = lambda t: np.sum(p * np.exp(t * m))
h = 1e-4
d1 = (M(h) - M(-h)) / (2 * h)
d2 = (M(h) - 2 * M(0) + M(-h)) / h**2
print(round(M(0), 3), round(d1, 3), round(d2, 3), round(d2 - d1**2, 3))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. By hand: $M'(t) = -0.2e^{-t} + 0.5e^{t}$ and $M''(t) = 0.2e^{-t} + 0.5e^{t}$.

In [ ]:
# check A (normalisation): M(0) = 1
assert abs(M(0) - 1) < 1e-12
print("check A passed")

In [ ]:
# check B (definition): the direct sum of (m - mu)^2 P(m) equals M''(0) - M'(0)^2
assert abs(((m - 0.3)**2 * p).sum() - (d2 - d1**2)) < 1e-6
print("check B passed")

**Your turn.** A count has $M(t) = (0.7 + 0.3e^t)^5$. Find its mean and variance. Which detector could give it?

*Hint:* differentiate with the chain rule; compare with the telescope of the lecture (Slide 15).

In [ ]:
# TODO: write your answer here

## Problem 6 · An MGF from data
500 hit times (simulated, mean 12.4 ns, $\sigma$ 0.40 ns). Estimate $M(t)$ by averaging $e^{tx}$, and read the mean off its slope at 0 with a step $h = 0.1$ per ns.

*Givens:* $\hat M(t) = \frac{1}{n}\sum e^{t x_i}$; seed 6.

**Predict.** Is $h = 0.1$ per ns small enough for the slope to be the mean? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
rng = np.random.default_rng(6)
x = rng.normal(12.4, 0.40, 500)                # ns
M = lambda t: np.mean(np.exp(t * x))
print(round((M(0.1) - 1) / 0.1, 1), round((M(0.001) - M(-0.001)) / 0.002, 3), round(x.max(), 2))
t = np.linspace(-0.5, 0.5, 21)
a, b, c = np.polyfit(t, np.log([M(s) for s in t]), 2)
print(round(b, 3), round(2 * a, 4), round(x.mean(), 3), round(x.var(), 4))

In [ ]:
plt.figure(figsize=(7, 3.5))
# take away the straight part b t: what is left is the parabola a t^2 = (variance / 2) t^2
plt.plot(t, np.log([M(s) for s in t]) - b * t, "o", label=r"$\ln \hat M(t) - b\,t$")
plt.plot(t, a * t**2 + c, label=r"$a\,t^2$ with $2a$ = variance")
plt.xlabel("t (per ns)"); plt.legend(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The first number is the forward step. Is a mean of 24.6 ns possible when no reading exceeds 13.65 ns?

In [ ]:
# check A (magnitude): a mean lies between the smallest and the largest reading
slope = (M(0.001) - M(-0.001)) / 0.002
assert x.min() <= slope <= x.max() and not (x.min() <= (M(0.1) - 1) / 0.1 <= x.max())
print("check A passed")

In [ ]:
# check B (independent method): the parabola of ln M gives the sample mean and variance
assert abs(b - x.mean()) < 0.01 and abs(2 * a - x.var()) < 0.01
print("check B passed")

**Your turn.** Use a central second difference of $\ln\hat M(t)$ at 0 with $h = 0.01$ per ns to estimate the variance. Compare with `x.var()`.

*Hint:* $[\ln\hat M(h) - 2\ln\hat M(0) + \ln\hat M(-h)]/h^2$, and $\ln\hat M(0) = 0$.

In [ ]:
# TODO: write your answer here

## Problem 7 · Sums: variances add
Each digitised reading has a rounding error uniform on $[-\tfrac12, \tfrac12]$ LSB. A sum of 12 readings carries 12 independent rounding errors. What is $\sigma$ of the total error?

*Givens:* one error: $\sigma = 1/\sqrt{12} = 0.289$ LSB; independent errors; seed 7.

**Predict.** Is it $12 \times 0.289 = 3.46$ LSB? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
rng = np.random.default_rng(7)
r = rng.uniform(-0.5, 0.5, (100_000, 12)).sum(axis=1)    # LSB
print(round(1 / np.sqrt(12), 3), round(12 / np.sqrt(12), 2))
print(round(r.std(), 3), round(np.sqrt(12 / 12), 3))
d = r - r.mean()
print(round(np.mean(d**3) / r.std()**3, 3), round(np.mean(d**4) / r.var()**2, 2))

In [ ]:
plt.figure(figsize=(7, 3.5))
z = np.linspace(-3, 3, 301)
plt.hist(r, bins=np.arange(-3, 3.001, 0.25), density=True, color="tab:blue", label="sum of 12 errors")
plt.plot(z, np.exp(-z**2 / 2) / np.sqrt(2 * np.pi), color="tab:green", label="Gaussian, σ = 1")
plt.xlabel("total rounding error (LSB)"); plt.legend(); plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. The first line has the trap; the second the simulation and the rule $\sqrt{n/12}$.

In [ ]:
# check A (simulation): sigma of 100 000 sums (seed 7) is 1.00 LSB to 0.01
assert abs(r.std() - 1.0) < 0.01
print("check A passed")

In [ ]:
# check B (MGF): M(t) = [sinh(t/2)/(t/2)]^12 has M''(0) = Var = 1
Mu = lambda t: (np.sinh(t / 2) / (t / 2))**12
h = 1e-3
print(round((Mu(h) - 2 + Mu(-h)) / h**2, 3))
assert abs((Mu(h) - 2 + Mu(-h)) / h**2 - 1) < 1e-3
print("check B passed")

**Your turn.** The difference of two digitised readings carries two rounding errors. What is $\sigma$ of the difference, and what shape does its histogram have?

*Hint:* a difference adds variances too; simulate with two uniform columns and plot a histogram.

In [ ]:
# TODO: write your answer here

## Problem 8 · A decay chain adds two lifetimes
A stopped π⁺ decays to a μ⁺ ($\tau_\pi = 26.033$ ns), which decays to e⁺ ($\tau_\mu = 2196.98$ ns). The time from the stop to the positron is $T_\pi + T_\mu$. Find its mean, $\sigma$ and skewness.

*Givens:* independent exponential stages; $M(t) = 1/[(1-\tau_\pi t)(1-\tau_\mu t)]$; seed 8.

**Predict.** Is $\sigma = \tau_\pi + \tau_\mu = 2223$ ns? Write your prediction in the next cell *before* you run anything below.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
tp, tm = 26.033, 2196.9811                     # ns: pion, muon mean lives
print(round(tp + tm, 1), round(np.hypot(tp, tm), 1))
print(round(2 * (tp**3 + tm**3) / np.hypot(tp, tm)**3, 4))
rng = np.random.default_rng(8)
T = rng.exponential(tp, 100_000) + rng.exponential(tm, 100_000)
print(round(T.mean()), round(T.std()), round(np.mean((T - T.mean())**3) / T.std()**3, 2))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which assumption was wrong. Which stage controls the spread? The pion's share of the variance is $\tau_\pi^2/(\tau_\pi^2 + \tau_\mu^2)$: compute it.

In [ ]:
# check A (limiting case): with tau_pi -> 0 the chain is the muon alone
assert abs(np.hypot(0.0, tm) - tm) < 1e-9 and abs(np.hypot(tp, tm) - tm) / tm < 1e-3
print("check A passed")

In [ ]:
# check B (simulation): 100 000 chains (seed 8) agree with sqrt(tau_pi^2 + tau_mu^2) to 1 %
assert abs(T.std() - np.hypot(tp, tm)) / np.hypot(tp, tm) < 0.01
print("check B passed")

**Your turn.** A chain has two stages with the same mean life $\tau$. Find the mean, $\sigma$ and skewness of the total time. Is it more or less skewed than one stage?

*Hint:* $M(t) = (1 - \tau t)^{-2}$; third central moments add.

In [ ]:
# TODO: write your answer here

## Before you close the tab
- Mistakes we met: $E[q^2] = E[q]^2$ (P1), a negative variance from cancellation (P2), "most above the mean" (P3), an integral stopped at $2\tau$ (P4), $\operatorname{Var} = M''(0)$ (P5), a step too large for the slope (P6), adding $\sigma$'s (P7, P8).
- Each has a check: a variance is never negative; a mean lies between the smallest and largest reading; $M(0) = 1$; a simulation with the same seed.
- **File → Save a copy in Drive** if you have not yet. Next week: Poisson and Bernoulli processes (Week 5).